# 1 Geocode Proper Addresses
Use the Google Geocoding API to validate/geocode addresses with Address_Type == 'Proper'.
Three geocoding strategies are provided:
1. **Simple** — street address only
2. **Improved** — full address (street + city + zip + state) with location metadata
3. **Strict** — state-level component restriction (zip removed for flexibility)

**Requires:** GOOGLE_MAPS_API_KEY in a .env file at the project root.

Input:  data/1_derived/5_geocode_truck_stops/4_final_coordinates.csv
Output: data/1_derived/5_geocode_truck_stops/5_geocoded_simple.csv
        data/1_derived/5_geocode_truck_stops/6_geocoded_strict.csv

In [ ]:
from pathlib import Path
import os
import time
import pandas as pd
import requests
from dotenv import load_dotenv


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'source').exists() and (candidate / 'temp').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
load_dotenv(PROJECT_ROOT / '.env')

IN_FILE  = PROJECT_ROOT / 'data' / '1_derived' / '5_geocode_truck_stops' / '4_final_coordinates.csv'
OUT_DIR  = PROJECT_ROOT / 'data' / '1_derived' / '5_geocode_truck_stops'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_SIMPLE = OUT_DIR / '5_geocoded_simple.csv'
OUT_STRICT = OUT_DIR / '6_geocoded_strict.csv'

API_KEY = os.getenv('GOOGLE_MAPS_API_KEY')
if not API_KEY:
    raise RuntimeError(
        'GOOGLE_MAPS_API_KEY not found. '
        'Create a .env file in the project root with: GOOGLE_MAPS_API_KEY=your_key_here'
    )

GEOCODE_URL = 'https://maps.googleapis.com/maps/api/geocode/json'
session = requests.Session()
_cache = {}

df = pd.read_csv(IN_FILE, low_memory=False)
proper_mask = df['Address_Type'] == 'Proper'
proper_indices = df[proper_mask].index.tolist()
print(f'Loaded {len(df):,} rows, {len(proper_indices):,} Proper addresses to geocode')
print(f'API Key loaded: {API_KEY[:8]}...{API_KEY[-4:]}')

In [ ]:
# --- Strategy 1: Simple geocoding (street address only) ---

def geocode_address(address: str, region: str = 'us') -> dict:
    if not isinstance(address, str) or not address.strip():
        return {'geo_status': 'EMPTY'}
    if address in _cache:
        return _cache[address]
    params = {'address': address, 'key': API_KEY, 'region': region}
    try:
        resp = session.get(GEOCODE_URL, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        result = {'geo_status': f'REQUEST_ERROR: {e}'}
        _cache[address] = result
        return result
    status = data.get('status', 'UNKNOWN')
    if status == 'OK' and data.get('results'):
        r0 = data['results'][0]
        result = {
            'geo_status': status,
            'formatted_address': r0.get('formatted_address'),
            'lat': r0['geometry']['location']['lat'],
            'lng': r0['geometry']['location']['lng'],
            'place_id': r0.get('place_id'),
            'types': ','.join(r0.get('types', [])),
        }
    else:
        result = {'geo_status': status, 'error_message': data.get('error_message')}
    _cache[address] = result
    time.sleep(0.2)
    return result


addr_col = 'address_standardized_OFF_parenthesis'
geo_cols = ['geo_status', 'formatted_address', 'lat', 'lng', 'place_id', 'types']
for col in geo_cols:
    df[f'Geocoding_API_{col}'] = None

print(f'Running simple geocoding on {len(proper_indices):,} addresses...')
start_time = time.time()
for i, idx in enumerate(proper_indices):
    if i % 50 == 0 and i > 0:
        elapsed = time.time() - start_time
        rate = i / (elapsed / 60)
        print(f'  {i}/{len(proper_indices)} done ({rate:.0f}/min)')
    result = geocode_address(df.loc[idx, addr_col])
    for col in geo_cols:
        df.loc[idx, f'Geocoding_API_{col}'] = result.get(col)

elapsed = time.time() - start_time
success = (df['Geocoding_API_geo_status'] == 'OK').sum()
print(f'\nSimple geocoding complete in {elapsed / 60:.1f} min')
print(f'Successful: {success:,}/{len(proper_indices):,} ({success / len(proper_indices) * 100:.1f}%)')

df.to_csv(OUT_SIMPLE, index=False)
print(f'Saved: {OUT_SIMPLE}')

In [ ]:
# --- Strategy 3: Strict geocoding (state-only component restriction) ---

_cache_strict = {}

def create_comprehensive_address(street, city, zip_code, state):
    parts = []
    if pd.notna(street) and str(street).strip():
        parts.append(str(street).strip())
    if pd.notna(city) and str(city).strip():
        parts.append(str(city).strip())
    sz = []
    if pd.notna(state) and str(state).strip():
        sz.append(str(state).strip())
    if pd.notna(zip_code) and str(zip_code).strip():
        sz.append(str(zip_code).strip())
    if sz:
        parts.append(' '.join(sz))
    return ', '.join(parts) if parts else ''


def geocode_address_strict(street, city=None, zip_code=None, state=None) -> dict:
    full_address = create_comprehensive_address(street, city, zip_code, state)
    if not full_address:
        return {'geo_status': 'EMPTY', 'input_address': ''}
    cache_key = f'strict_{full_address}_{state}'
    if cache_key in _cache_strict:
        r = _cache_strict[cache_key].copy()
        r['input_address'] = full_address
        return r
    components = ['country:US']
    if pd.notna(state) and str(state).strip():
        components.append(f'administrative_area_level_1:{str(state).strip().upper()}')
    params = {
        'address': full_address, 'key': API_KEY, 'region': 'us',
        'components': '|'.join(components)
    }
    try:
        resp = session.get(GEOCODE_URL, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        result = {'geo_status': f'REQUEST_ERROR: {e}', 'input_address': full_address}
        _cache_strict[cache_key] = result
        return result
    status = data.get('status', 'UNKNOWN')
    if status == 'OK' and data.get('results'):
        r0 = data['results'][0]
        result_state = None
        for comp in r0.get('address_components', []):
            if 'administrative_area_level_1' in comp.get('types', []):
                result_state = comp.get('short_name')
        result = {
            'geo_status': status, 'input_address': full_address,
            'formatted_address': r0.get('formatted_address'),
            'lat': r0['geometry']['location']['lat'],
            'lng': r0['geometry']['location']['lng'],
            'place_id': r0.get('place_id'),
            'types': ','.join(r0.get('types', [])),
            'location_type': r0['geometry'].get('location_type'),
            'partial_match': r0.get('partial_match', False),
            'result_state': result_state,
            'input_state': str(state).strip().upper() if pd.notna(state) else None,
            'state_match': result_state == str(state).strip().upper() if pd.notna(state) and result_state else None,
        }
    else:
        result = {'geo_status': status, 'input_address': full_address,
                  'error_message': data.get('error_message')}
    _cache_strict[cache_key] = result
    time.sleep(0.2)
    return result


df_strict = df.copy()
strict_cols = ['geo_status', 'input_address', 'formatted_address', 'lat', 'lng',
               'place_id', 'types', 'location_type', 'partial_match',
               'result_state', 'input_state', 'state_match']
for col in strict_cols:
    df_strict[f'Strict_Geocoding_{col}'] = None

print(f'Running strict geocoding on {len(proper_indices):,} addresses...')
start_time = time.time()
for i, idx in enumerate(proper_indices):
    if i % 50 == 0 and i > 0:
        elapsed = time.time() - start_time
        rate = i / (elapsed / 60)
        print(f'  {i}/{len(proper_indices)} done ({rate:.0f}/min)')
    row = df.loc[idx]
    result = geocode_address_strict(
        row['address_standardized_OFF_parenthesis'],
        row['city'], row['zip_code'], row['state']
    )
    for col in strict_cols:
        df_strict.loc[idx, f'Strict_Geocoding_{col}'] = result.get(col)

elapsed = time.time() - start_time
success = (df_strict['Strict_Geocoding_geo_status'] == 'OK').sum()
print(f'\nStrict geocoding complete in {elapsed / 60:.1f} min')
print(f'Successful: {success:,}/{len(proper_indices):,} ({success / len(proper_indices) * 100:.1f}%)')

# State match analysis
ok_mask = df_strict['Strict_Geocoding_geo_status'] == 'OK'
state_matches = (df_strict.loc[ok_mask, 'Strict_Geocoding_state_match'] == True).sum()
print(f'State match rate: {state_matches}/{ok_mask.sum()} ({state_matches / ok_mask.sum() * 100:.1f}%)')

df_strict.to_csv(OUT_STRICT, index=False)
print(f'Saved: {OUT_STRICT}')